# 04 — Streaming Analytics

Queries the Silver table to show real-time gaming floor analytics.
These are the same queries we'll run in Part 2 (Zerobus) for an
apples-to-apples comparison.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "streaming_demo")

SILVER_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.slot_events_silver"

## Revenue by Casino Floor

How much is each floor generating? (House revenue = total bets - total wins)

In [2]:
spark.sql(f"""
    SELECT
        casino_floor,
        COUNT(*) AS total_plays,
        ROUND(SUM(bet_amount), 2) AS total_bets,
        ROUND(SUM(win_amount), 2) AS total_payouts,
        ROUND(SUM(bet_amount) - SUM(win_amount), 2) AS house_revenue
    FROM {SILVER_TABLE}
    GROUP BY casino_floor
    ORDER BY house_revenue DESC
""").show(truncate=False)

+------------+-----------+----------+-------------+-------------+
|casino_floor|total_plays|total_bets|total_payouts|house_revenue|
+------------+-----------+----------+-------------+-------------+
|VIP Lounge  |93         |12769.03  |8084.36      |4684.67      |
|Main Floor  |98         |13988.02  |15662.27     |-1674.25     |
|Sports Zone |104        |15632.78  |17938.36     |-2305.58     |
|Penny Lane  |103        |16960.18  |21633.78     |-4673.6      |
|High Roller |102        |15213.66  |32613.89     |-17400.23    |
+------------+-----------+----------+-------------+-------------+



## Top 10 Machines by Play Volume

In [3]:
spark.sql(f"""
    SELECT
        machine_id,
        casino_floor,
        game_type,
        COUNT(*) AS total_plays,
        ROUND(SUM(bet_amount), 2) AS total_wagered
    FROM {SILVER_TABLE}
    GROUP BY machine_id, casino_floor, game_type
    ORDER BY total_plays DESC
    LIMIT 10
""").show(truncate=False)

+----------+------------+--------------------+-----------+-------------+
|machine_id|casino_floor|game_type           |total_plays|total_wagered|
+----------+------------+--------------------+-----------+-------------+
|MCH-0050  |Sports Zone |Electronic Blackjack|3          |1378.7       |
|MCH-0050  |Main Floor  |Slots               |3          |75.27        |
|MCH-0004  |Sports Zone |Electronic Blackjack|3          |649.01       |
|MCH-0042  |Main Floor  |Electronic Blackjack|3          |1722.18      |
|MCH-0028  |Sports Zone |Electronic Roulette |3          |600.87       |
|MCH-0037  |High Roller |Electronic Roulette |3          |848.36       |
|MCH-0023  |Sports Zone |Electronic Roulette |3          |1151.77      |
|MCH-0029  |Main Floor  |Keno                |3          |38.6         |
|MCH-0035  |VIP Lounge  |Keno                |3          |21.86        |
|MCH-0014  |High Roller |Keno                |3          |8.14         |
+----------+------------+--------------------+-----

## Win/Loss Distribution by Game Type

In [4]:
spark.sql(f"""
    SELECT
        game_type,
        COUNT(*) AS total_plays,
        SUM(CASE WHEN is_win THEN 1 ELSE 0 END) AS wins,
        SUM(CASE WHEN NOT is_win THEN 1 ELSE 0 END) AS losses,
        ROUND(SUM(CASE WHEN is_win THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS win_pct,
        ROUND(AVG(bet_amount), 2) AS avg_bet,
        ROUND(AVG(CASE WHEN is_win THEN win_amount END), 2) AS avg_win_payout
    FROM {SILVER_TABLE}
    GROUP BY game_type
    ORDER BY total_plays DESC
""").show(truncate=False)

+--------------------+-----------+----+------+-------+-------+--------------+
|game_type           |total_plays|wins|losses|win_pct|avg_bet|avg_win_payout|
+--------------------+-----------+----+------+-------+-------+--------------+
|Keno                |112        |54  |58    |48.2   |9.97   |34.58         |
|Electronic Roulette |101        |44  |57    |43.6   |270.19 |509.12        |
|Video Poker         |100        |44  |56    |44.0   |48.92  |167.64        |
|Slots               |99         |39  |60    |39.4   |23.56  |60.9          |
|Electronic Blackjack|88         |42  |46    |47.7   |442.43 |1474.11       |
+--------------------+-----------+----+------+-------+-------+--------------+



## Bet Tier Breakdown

How does play behavior break down across Low / Medium / High / Whale tiers?

In [5]:
spark.sql(f"""
    SELECT
        bet_tier,
        COUNT(*) AS total_plays,
        COUNT(DISTINCT player_id) AS unique_players,
        ROUND(SUM(bet_amount), 2) AS total_wagered,
        ROUND(SUM(bet_amount) - SUM(win_amount), 2) AS house_revenue
    FROM {SILVER_TABLE}
    GROUP BY bet_tier
    ORDER BY total_wagered DESC
""").show(truncate=False)

+--------+-----------+--------------+-------------+-------------+
|bet_tier|total_plays|unique_players|total_wagered|house_revenue|
+--------+-----------+--------------+-------------+-------------+
|High    |122        |73            |37569.77     |-3689.94     |
|Whale   |36         |28            |26997.75     |-13345.64    |
|Medium  |244        |97            |9481.39      |-3769.18     |
|Low     |98         |62            |514.76       |-564.23      |
+--------+-----------+--------------+-------------+-------------+

